## Merge

In [1]:
import dask.dataframe as dd
import os

linked_features = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']

files_info_processed = [
    {
        "name": "processed_cultures_cohort",
        "path": "microbiology_cultures_cohort.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "order_time_jittered_utc":"object",
            "organism": "int8",
            "antibiotic": "int8",
            "susceptibility": "int8"
        }
    },
    {
        "name": "processed_prior_med",
        "path": "microbiology_cultures_prior_med.csv",
        "merge_on": linked_features[:3], # "order_time_jittered_utc": "object", this column is not used for merging in this file, we will explane this later in the last of this notebook
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "min_time_to_med": "Int32", 
            "medication_category": "int8",
            "n_med_categories": "int32",
            "med_within_30_days": "boolean"
        }
    },
    {
        "name": "processed_cultures_demographics",
        "path": "microbiology_cultures_demographics.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "age": "int8",
            "gender": "int8"
        }
    },
     {
        "name": "processed_cultures_labs",
        "path": "microbiology_cultures_labs.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_wbc": "float32",
            "median_neutrophils": "float32",
            "median_lymphocytes": "float32",
            "median_hgb": "float32",
            "median_plt": "float32",
            "median_na": "float32",
            "median_hco3": "float32",
            "median_bun": "float32",
            "median_cr": "float32",
            "median_lactate": "float32",
            "median_procalcitonin": "float32",
            "delta_wbc": "float32",
            "delta_na": "float32",
            "delta_hco3": "float32",
            "delta_bun": "float32",
            "delta_cr": "float32"
        }
    },
    {
        "name": "processed_cultures_vitals",
        "path": "microbiology_cultures_vitals.csv",
        "merge_on": linked_features[:3],
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "first_heartrate": "float32",
            "first_resprate": "float32",
            "first_temp": "float32",
            "first_sysbp": "float32",
            "first_diasbp": "float32",
            "last_heartrate": "float32",
            "last_resprate": "float32",
            "last_temp": "float32",
            "last_sysbp": "float32",
            "last_diasbp": "float32",
            "median_heartrate": "float32",
            "median_resprate": "float32",
            "median_temp": "float32",
            "median_sysbp": "float32",
            "median_diasbp": "float32",
            "delta_heartrate": "float32",
            "delta_resprate": "float32",
            "delta_temp": "float32",
            "delta_sysbp": "float32",
            "delta_diasbp": "float32"
        }
    },
     {
        "name": "processed_antibiotic_class_exposure",
        "path": "microbiology_cultures_antibiotic_class_exposure.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "antibiotic_class": "int8",
            "min_time_to_class": "Int32"  
        }
    },
    {
    "name": "processed_cultures_comorbidity",
    "path": "microbiology_cultures_comorbidity.csv",
    "merge_on": linked_features, 
    "dtype": {
        "anon_id": "object",
        "pat_enc_csn_id_coded": "int64",
        "order_proc_id_coded": "int64",
        "order_time_jittered_utc": "object",
        "has_diabetes": "int8",
        "has_chf": "int8",
        "has_ckd": "int8",
        "has_cancer": "int8",
        "has_transplant": "int8",
        "has_immunosuppression": "int8",
        "has_copd": "int8",
        "has_liver_disease": "int8",
        "has_dementia": "int8",
        "has_stroke": "int8",
        "has_other_comorbid": "int8"
    }
}

]

sample_folder = 'new_sample_one_processed/'
merged_folder = 'merged_sample_one_processed/'
os.makedirs(merged_folder, exist_ok=True)         

file_df_pairs = [file['name'] + '.parquet' for file in files_info_processed]

based_df_dtype = files_info_processed[0]['dtype']
df_dtype = files_info_processed[1]['dtype']

based_df = dd.read_parquet(sample_folder + file_df_pairs[0], 
                           dtype = based_df_dtype)
print(f"Based DF Info: {based_df.compute().info()}")
df = dd.read_parquet(sample_folder + file_df_pairs[1], 
                           dtype = df_dtype )
df = df.drop(columns=["order_time_jittered_utc"], errors="ignore") # this column is not used for merging in this file, we will explane this later in the last of this notebook

print(f"df Info: {df.compute().info()}")

based_df = based_df.merge(df, how="left", on=files_info_processed[1]['merge_on'])

print(f"Start merging: left {files_info_processed[0]['name']}.parquet ; right {files_info_processed[1]['name']}.parquet")

print(f"Merged DF Info: {based_df.compute().info()}")

based_df.to_parquet(
            merged_folder + "stage01/",
            engine='pyarrow',
            compression='snappy',
            write_index=False
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

<class 'pandas.DataFrame'>
RangeIndex: 29632 entries, 0 to 29631
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   anon_id                  29632 non-null  string
 1   pat_enc_csn_id_coded     29632 non-null  int64 
 2   order_proc_id_coded      29632 non-null  int64 
 3   order_time_jittered_utc  29632 non-null  string
 4   organism                 29632 non-null  int8  
 5   antibiotic               29632 non-null  int8  
 6   susceptibility           25855 non-null  Int8  
dtypes: Int8(1), int64(2), int8(2), string(2)
memory usage: 2.0 MB
Based DF Info: None
<class 'pandas.DataFrame'>
RangeIndex: 3513 entries, 0 to 3512
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   anon_id               3513 non-null   string 
 1   pat_enc_csn_id_coded  3513 non-null   int64  
 2   order_proc_id_coded   3513 non-null

In [2]:

df_dtype = files_info_processed[2]['dtype']
based_df = dd.read_parquet(merged_folder + "stage01/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[2], 
                           dtype = df_dtype )
# print(based_df.info())  # Check the structure before merging

based_df = based_df.merge(df, how='left', on=files_info_processed[2]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[2]['name']}.parquet")

based_df.to_parquet(
    merged_folder + "stage02/",
    engine="pyarrow",
    compression="snappy",
    write_index=False,
    overwrite=True
)

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_demographics.parquet
Merged result saved - Cols # 12


In [3]:

df_dtype = files_info_processed[3]['dtype']
based_df = dd.read_parquet(merged_folder + "stage02/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[3], 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[3]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[3]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage03/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_labs.parquet
Merged result saved - Cols # 28


In [4]:

df_dtype = files_info_processed[4]['dtype']
based_df = dd.read_parquet(merged_folder + "stage03/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[4], 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[4]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[4]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage04/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_vitals.parquet
Merged result saved - Cols # 48


## check the merge result until now!

In [5]:
based_df = dd.read_parquet(merged_folder + "stage04/" +"*.parquet", 
                           dtype = based_df_dtype)
num_cols = len(based_df.columns)
num_rows = based_df.shape[0].compute()
print(f"Shape: ({num_rows},{num_cols})")

Shape: (540887,48)


In [6]:
import dask.dataframe as dd
import os

linked_features = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']

files_info_processed = [
    {
        "name": "processed_cultures_cohort",
        "path": "microbiology_cultures_cohort.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "order_time_jittered_utc":"object",
            "organism": "int8",
            "antibiotic": "int8",
            "susceptibility": "int8"
        }
    },
    {
        "name": "processed_prior_med",
        "path": "microbiology_cultures_prior_med.csv",
        "merge_on": linked_features[:3], # "order_time_jittered_utc": "object", this column is not used for merging in this file, we will explane this later in the last of this notebook
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "min_time_to_med": "Int32", 
            "medication_category": "int8",
            "n_med_categories": "int32",
            "med_within_30_days": "boolean"
        }
    },
    {
        "name": "processed_cultures_demographics",
        "path": "microbiology_cultures_demographics.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "age": "int8",
            "gender": "int8"
        }
    },
     {
        "name": "processed_cultures_labs",
        "path": "microbiology_cultures_labs.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_wbc": "float32",
            "median_neutrophils": "float32",
            "median_lymphocytes": "float32",
            "median_hgb": "float32",
            "median_plt": "float32",
            "median_na": "float32",
            "median_hco3": "float32",
            "median_bun": "float32",
            "median_cr": "float32",
            "median_lactate": "float32",
            "median_procalcitonin": "float32",
            "delta_wbc": "float32",
            "delta_na": "float32",
            "delta_hco3": "float32",
            "delta_bun": "float32",
            "delta_cr": "float32"
        }
    },
    {
        "name": "processed_cultures_vitals",
        "path": "microbiology_cultures_vitals.csv",
        "merge_on": linked_features[:3],
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "first_heartrate": "float32",
            "first_resprate": "float32",
            "first_temp": "float32",
            "first_sysbp": "float32",
            "first_diasbp": "float32",
            "last_heartrate": "float32",
            "last_resprate": "float32",
            "last_temp": "float32",
            "last_sysbp": "float32",
            "last_diasbp": "float32",
            "median_heartrate": "float32",
            "median_resprate": "float32",
            "median_temp": "float32",
            "median_sysbp": "float32",
            "median_diasbp": "float32",
            "delta_heartrate": "float32",
            "delta_resprate": "float32",
            "delta_temp": "float32",
            "delta_sysbp": "float32",
            "delta_diasbp": "float32"
        }
    },
     {
        "name": "processed_antibiotic_class_exposure",
        "path": "microbiology_cultures_antibiotic_class_exposure.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "antibiotic_class": "int8",
            "min_time_to_class": "Int32"  
        }
    },
    {
    "name": "processed_cultures_comorbidity",
    "path": "microbiology_cultures_comorbidity.csv",
    "merge_on": linked_features, 
    "dtype": {
        "anon_id": "object",
        "pat_enc_csn_id_coded": "int64",
        "order_proc_id_coded": "int64",
        "order_time_jittered_utc": "object",
        "has_diabetes": "int8",
        "has_chf": "int8",
        "has_ckd": "int8",
        "has_cancer": "int8",
        "has_transplant": "int8",
        "has_immunosuppression": "int8",
        "has_copd": "int8",
        "has_liver_disease": "int8",
        "has_dementia": "int8",
        "has_stroke": "int8",
        "has_other_comorbid": "int8"
    }
}

]

In [7]:
import dask.dataframe as dd
import os

sample_folder = 'new_sample_one_processed/'
merged_folder = 'merged_sample_one_processed/'
os.makedirs(merged_folder, exist_ok=True)  

based_df_dtype = {}
for f in files_info_processed[:5]:
    based_df_dtype.update(f['dtype'])

df_dtype = files_info_processed[5]['dtype']
based_df = dd.read_parquet(merged_folder + "stage04/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + files_info_processed[5]['name'] + '.parquet', 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[5]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[5]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage05/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_antibiotic_class_exposure.parquet
Merged result saved - Cols # 50


In [8]:

df_dtype = files_info_processed[6]['dtype']
based_df = dd.read_parquet(merged_folder + "stage05/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + files_info_processed[6]['name'] + '.parquet', 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[6]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[6]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "final_stage/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_comorbidity.parquet
Merged result saved - Cols # 61


# Explain why "order_time_jittered_utc" not used for merging:

Even after checking the raw data, both `microbiology_cultures_prior_med.csv` and `microbiology_cultures_cohort.csv` contain the `order_time_jittered_utc` column; however, there is **zero overlap on the full set of merge keys when this timestamp is included**. In other words, the intersection on (`anon_id`, `pat_enc_csn_id_coded`, `order_proc_id_coded`, `order_time_jittered_utc`) is 0, which indicates that `order_time_jittered_utc` is not a consistent linkage variable between these two tables (e.g., due to independent jittering or differing timestamp definitions). Therefore, we excluded `order_time_jittered_utc` from the merge keys and merged using the three stable identifiers only.

In [9]:
import pandas as pd

raw_data_folder = 'doi_10_5061_dryad_jq2bvq8kp__v20250411/'
meds = pd.read_csv(
    raw_data_folder + "microbiology_cultures_prior_med.csv",
    usecols=["anon_id","pat_enc_csn_id_coded",
             "order_proc_id_coded","order_time_jittered_utc"]
    )
meds.info()

<class 'pandas.DataFrame'>
RangeIndex: 9823458 entries, 0 to 9823457
Data columns (total 4 columns):
 #   Column                   Dtype
---  ------                   -----
 0   anon_id                  str  
 1   pat_enc_csn_id_coded     int64
 2   order_proc_id_coded      int64
 3   order_time_jittered_utc  str  
dtypes: int64(2), str(2)
memory usage: 596.1 MB


In [10]:
cohort = pd.read_csv(
    raw_data_folder + "microbiology_cultures_cohort.csv",
    usecols=["anon_id","pat_enc_csn_id_coded",
             "order_proc_id_coded","order_time_jittered_utc"]
)
cohort.info()

<class 'pandas.DataFrame'>
RangeIndex: 2241050 entries, 0 to 2241049
Data columns (total 4 columns):
 #   Column                   Dtype
---  ------                   -----
 0   anon_id                  str  
 1   pat_enc_csn_id_coded     int64
 2   order_proc_id_coded      int64
 3   order_time_jittered_utc  str  
dtypes: int64(2), str(2)
memory usage: 140.4 MB


In [11]:
for d in [cohort, meds]:
    d["order_time_jittered_utc"] = (
        d["order_time_jittered_utc"]
        .astype(str)
        .str.strip()
    )

In [12]:
k4 = ["anon_id","pat_enc_csn_id_coded",
      "order_proc_id_coded","order_time_jittered_utc"]

cohort_k4 = cohort[k4].drop_duplicates()
meds_k4   = meds[k4].drop_duplicates()

intersection_k4 = cohort_k4.merge(meds_k4, how="inner", on=k4)

print("RAW intersection on 4 keys:", len(intersection_k4))

RAW intersection on 4 keys: 0
